# 03 Merge: OpenCity + MyNeta

Joins the OpenCity results (general elections and by-elections) with the cleaned MyNeta affidavits from 02b. The TCPD party table is used to line up party names. Outputs go to `clean/`.

In [1]:
from pathlib import Path
import re
import difflib

import numpy as np
import pandas as pd

COLLECTION_DATE = "2026-10-20"
STATES = ["Bihar", "Maharashtra", "Uttar Pradesh", "Tamil Nadu"]

def find_file(name):
    for folder in [Path("../clean"), Path("../interim"), Path("clean"), Path(".")]:
        if (folder / name).exists():
            return folder / name
    raise FileNotFoundError(name)

clean_dir = Path("../clean") if Path("../clean").is_dir() else Path("clean")

results = pd.read_csv(find_file(f"opencity_results_clean_{COLLECTION_DATE}.csv"))
oc_bye = pd.read_csv(find_file("opencity_byelections_2014_onwards.csv"), low_memory=False)
myneta = pd.read_csv(find_file(f"myneta_candidates_clean_{COLLECTION_DATE}.csv"),
                     dtype={"age": "Int64"}, parse_dates=["election_date", "bye_election_date"])
ppi = pd.read_csv(find_file("TCPD-PPI_1962_2021.csv"))

# row numbers of the input files, kept so every output row can be traced back
results["OC_Row"] = results.index
oc_bye["OC_Row"] = oc_bye.index
myneta["MN_Row"] = myneta.index

print("OpenCity:", results.shape, "| by-elections (all states):", oc_bye.shape, "| MyNeta:", myneta.shape, "| TCPD:", ppi.shape)

OpenCity: (10378, 24) | by-elections (all states): (410, 46) | MyNeta: (10395, 24) | TCPD: (10153, 21)


### Names

Upper case, symbols to spaces, titles (`Dr`, `Adv`, `Smt`...) dropped. Two keys for matching: words sorted (`Name_Key`) and letters sorted (`Letters_Key`, so `K P MUNUSAMY` = `KP MUNUSAMY`).

In [2]:
TITLES = {"DR", "PROF", "PROFESSOR", "ADV", "ADVOCATE", "ER", "ENGR", "SHRI", "SRI", "SHREE", "SMT", "SMTI", "SHRIMATI",
          "TMT", "THIRU", "THIRUMATHI", "THIRUVALAR", "SELVI", "MR", "MRS", "MS", "MISS", "COL", "CAPT", "MAJ", "RETD", "RTD"}

def clean_name(name):
    text = re.sub(r"[^A-Z0-9 ]", " ", str(name).upper())
    return " ".join(t for t in text.split() if t not in TITLES)

def add_name_keys(frame, col):
    frame["Name"] = frame[col].map(clean_name)
    frame["Name_Key"] = frame["Name"].map(lambda n: " ".join(sorted(n.split())))
    frame["Letters_Key"] = frame["Name"].map(lambda n: "".join(sorted(n.replace(" ", ""))))
    return frame

myneta = add_name_keys(myneta, "candidate")
results = add_name_keys(results, "Candidate")

### Seat numbers for MyNeta

MyNeta only has the seat name (`GAYA (SC)`). Names are compared without `(SC)`/`(ST)` and symbols; the few left over are matched by close spelling.

In [3]:
def seat_norm(s):
    s = re.sub(r"\((SC|ST|GEN)\)", "", str(s), flags=re.I)
    return re.sub(r"[^A-Z0-9]", "", s.upper())

seats = results.drop_duplicates(["State", "Constituency_No"])[["State", "Constituency_No", "Constituency_Name"]]
seat_lookup = {(r.State, seat_norm(r.Constituency_Name)): r.Constituency_No for r in seats.itertuples()}
assert len(seat_lookup) == len(seats)

myneta["State"] = myneta["state"].str.title()
assert set(myneta["State"]) <= set(STATES)

resolved, fuzzy_seats = {}, []
for state, const in myneta[["State", "const"]].drop_duplicates().itertuples(index=False):
    key = (state, seat_norm(const))
    if key not in seat_lookup:
        same_state = [k[1] for k in seat_lookup if k[0] == state]
        key = (state, difflib.get_close_matches(key[1], same_state, n=1, cutoff=0.8)[0])
        fuzzy_seats.append((const, key[1]))
    resolved[(state, const)] = seat_lookup[key]

myneta["Constituency_No"] = [resolved[(s, c)] for s, c in zip(myneta["State"], myneta["const"])]
print("seat names:", len(resolved), "| matched by close spelling:", fuzzy_seats)

seat names: 255 | matched by close spelling: [('AHMEDNAGAR', 'AHMADNAGAR'), ('HATKANANGALE', 'HATKANANGLE'), ('TIRUVALLUR (SC)', 'THIRUVALLUR'), ('TIRUVALLUR', 'THIRUVALLUR'), ('GAUTAM BUDHHA NAGAR', 'GAUTAMBUDDHANAGAR')]


### Parties

Each party string is looked up among the TCPD Lok Sabha names and abbreviations (same state first). `Party_Method` says how: exact, exact-old (party stopped before 2014), alias (hand rule), ambiguous (largest party taken), fuzzy (95% spelling) or unmapped (not in TCPD, mostly parties formed after 2021).

In [4]:
def pnorm(text):
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())

ppi_ls = ppi[ppi["Assembly"] == "Lok_Sabha"].copy()
ppi_ls["State"] = ppi_ls["State_Name"].str.replace("_", " ")
ppi_ls["Party_Name"] = ppi_ls["Party_Name"].str.replace(r"\s+", " ", regex=True).str.strip()

party_info = ppi_ls.groupby("Party_ID").agg(Party_Name=("Party_Name", "first"), Party_Type=("Party_Type", "first"),
                                            Last_Year=("Last_Year", "max"), Size=("Candidates_Contested", "max"))

name_lookup, abbr_lookup = {}, {}
for r in ppi_ls.itertuples():
    name_lookup.setdefault(pnorm(r.Party_Name), set()).add((r.Party_ID, r.State))
    for text in [r.Frequent_Abbreviation, r.Last_Abbreviation] + str(r.Abbreviations).split("|"):
        key = pnorm(text)
        if key and key != "NAN":
            abbr_lookup.setdefault(key, set()).add((r.Party_ID, r.State))
lookup = {**abbr_lookup, **name_lookup}   # full name wins over an abbreviation

ALIASES = {"AIADMK": "ADMK", "AAP": "AAAP"}
FULL_NAMES = {
    "Bharatiya Janata Party": "BJP", "Indian National Congress": "INC", "Bahujan Samaj Party": "BSP",
    "Samajwadi Party": "SP", "Rashtriya Janata Dal": "RJD", "Janata Dal (United)": "JD(U)",
    "Dravida Munnetra Kazhagam": "DMK", "All India Anna Dravida Munnetra Kazhagam": "ADMK",
    "Nationalist Congress Party": "NCP", "Communist Party of India": "CPI", "Communist Party of India (Marxist)": "CPM",
    "Communist Party of India (Marxist-Leninist) (Liberation)": "CPI(ML)(L)", "Shiv Sena": "SHS",
    "All India Majlis-E-Ittehadul Muslimeen": "AIMIM", "Pattali Makkal Katchi": "PMK",
    "Desiya Murpokku Dravida Kazhagam": "DMDK", "Viduthalai Chiruthaigal Katchi": "VCK",
    "Marumalarchi Dravida Munnetra Kazhagam": "MDMK", "Rashtriya Lok Dal": "RLD", "Naam Tamilar Katchi": "NTK",
    "Indian Union Muslim League": "IUML", "Apna Dal (Soneylal)": "ADAL", "Hindustani Awam Morcha (Secular)": "HAMS",
    "Lok Janshakti Party": "LJP", "Independent": "IND", "None of the Above": "NOTA",
}
ALIASES.update({pnorm(name): pnorm(c) for name, c in FULL_NAMES.items()})
# 2022/23 splits go to the parent party
FACTION_RULES = [("UDDHAV", "SHS"), ("SHARADCHANDRA", "NCP"), ("NCPSP", "NCP"), ("RAMVILAS", "LJP"), ("LJPRV", "LJP")]

def resolve_party(raw, state):
    key, method = pnorm(raw), "exact"
    for fragment, parent in FACTION_RULES:
        if fragment in key:
            key, method = parent, "alias"
    if key in ALIASES:
        key, method = ALIASES[key], "alias"
    options = lookup.get(key)
    if not options:
        close = difflib.get_close_matches(key, list(lookup), n=1, cutoff=0.95)
        if not close:
            return np.nan, "unmapped"
        options, method = lookup[close[0]], "fuzzy"
    ids = {pid for pid, st in options if st in (state, "All States")} or {pid for pid, _ in options}
    live = [p for p in ids if party_info.loc[p, "Last_Year"] >= 2014]
    if len(live) == 1:
        return live[0], method
    if len(live) > 1:
        return max(live, key=lambda p: party_info.loc[p, "Size"]), "ambiguous"
    best = max(ids, key=lambda p: party_info.loc[p, "Size"])
    return best, ("exact-old" if method == "exact" else method)

In [5]:
results["Party_Raw"] = results["Party_Full"]
myneta["Party_Raw"] = myneta["party"]
oc_bye = oc_bye.rename(columns={"Party_ID": "Party_ID_in_file"})
oc_bye["State"] = oc_bye["State_Name"].str.replace("_", " ")
oc_bye["Party_Raw"] = oc_bye["Party"]

pairs = pd.concat([f[["State", "Party_Raw"]] for f in [results, myneta, oc_bye]]).drop_duplicates()
party_map = pairs.copy()
party_map[["Party_ID", "Party_Method"]] = [resolve_party(raw, state) for state, raw in pairs.itertuples(index=False)]
party_map["Party_Canonical"] = party_map["Party_ID"].map(party_info["Party_Name"])
party_map["Party_Type"] = party_map["Party_ID"].map(party_info["Party_Type"]).fillna("Not in TCPD")
print(party_map["Party_Method"].value_counts().to_dict())

def add_party(frame):
    out = frame.merge(party_map, on=["State", "Party_Raw"], how="left", validate="many_to_one")
    out.index = frame.index
    return out

results, myneta, oc_bye = add_party(results), add_party(myneta), add_party(oc_bye)

{'exact': 1327, 'unmapped': 276, 'alias': 54, 'ambiguous': 38, 'fuzzy': 32, 'exact-old': 8}


### Matching candidates

Inside each seat and election, rows are paired one to one, rule by rule:

1. same `Name_Key`
2. same `Letters_Key`
3. initials + surname on one side, full name on the other
4. name similarity >= 0.85
5. same party (not IND) and similarity >= 0.70
6. last one left on both sides, same party
7. the winner on both sides, if still unpaired

Rules 1 to 5 are skipped when the two parties differ and one of them is a national or state party.

In [6]:
IND_ID = party_info.index[party_info["Party_Type"] == "Independents"][0]
BIG_TYPES = {"National Party", "State-based Party"}

def similarity(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio()

def same_party(o, m):
    return pd.notna(o) and pd.notna(m) and o == m and o != IND_ID

def party_conflict(o, m):
    if pd.isna(o) or pd.isna(m) or o == m:
        return False
    return bool({party_info.at[o, "Party_Type"], party_info.at[m, "Party_Type"]} & BIG_TYPES)

def initials_match(short, long):
    ts, tl = short.split(), long.split()
    full = [t for t in ts if len(t) > 1]
    initials = [t for t in ts if len(t) == 1]
    if not full or not initials or len(tl) < len(ts):
        return False
    rest = list(tl)
    for t in full:
        best = max(rest, key=lambda w: similarity(t, w))
        if similarity(t, best) < 0.85:
            return False
        rest.remove(best)
    firsts = [w[0] for w in rest]
    for i in initials:
        if i not in firsts:
            return False
        firsts.remove(i)
    return True

def match_seat(oc, mn):
    pairs, used_oc, used_mn = [], set(), set()
    def take(o, m, rule, score):
        pairs.append((o, m, rule, score)); used_oc.add(o); used_mn.add(m)

    # rules 1-2
    for rule, col in [(1, "Name_Key"), (2, "Letters_Key")]:
        for m in mn.index:
            if m in used_mn:
                continue
            found = [o for o in oc.index if o not in used_oc and oc.at[o, col] == mn.at[m, col]
                     and not party_conflict(oc.at[o, "Party_ID"], mn.at[m, "Party_ID"])]
            if found:
                preferred = [o for o in found if pd.notna(oc.at[o, "Party_ID"]) and oc.at[o, "Party_ID"] == mn.at[m, "Party_ID"]]
                take((preferred or found)[0], m, rule, 1.0)

    # rules 3-5, best score first
    scored = []
    for m in mn.index:
        for o in oc.index:
            if m in used_mn or o in used_oc or party_conflict(oc.at[o, "Party_ID"], mn.at[m, "Party_ID"]):
                continue
            ratio = similarity(oc.at[o, "Name_Key"], mn.at[m, "Name_Key"])
            party_ok = same_party(oc.at[o, "Party_ID"], mn.at[m, "Party_ID"])
            a, b = oc.at[o, "Name"], mn.at[m, "Name"]
            if initials_match(a, b) or initials_match(b, a):
                scored.append((0.95 + 0.05 * party_ok, o, m, 3, max(ratio, 0.95)))
            elif ratio >= 0.85:
                scored.append((ratio + 0.05 * party_ok, o, m, 4, ratio))
            elif party_ok and ratio >= 0.70:
                scored.append((ratio, o, m, 5, ratio))
    for _, o, m, rule, ratio in sorted(scored, reverse=True):
        if o not in used_oc and m not in used_mn:
            take(o, m, rule, ratio)

    # rule 6
    left_oc = [o for o in oc.index if o not in used_oc]
    left_mn = [m for m in mn.index if m not in used_mn]
    if len(left_oc) == 1 and len(left_mn) == 1 and same_party(oc.at[left_oc[0], "Party_ID"], mn.at[left_mn[0], "Party_ID"]):
        take(left_oc[0], left_mn[0], 6, similarity(oc.at[left_oc[0], "Name_Key"], mn.at[left_mn[0], "Name_Key"]))

    # rule 7
    win_oc = [o for o in oc.index if o not in used_oc and oc.at[o, "Winner"]]
    win_mn = [m for m in mn.index if m not in used_mn and mn.at[m, "is_winner"]]
    if len(win_oc) == 1 and len(win_mn) == 1 and not party_conflict(oc.at[win_oc[0], "Party_ID"], mn.at[win_mn[0], "Party_ID"]):
        take(win_oc[0], win_mn[0], 7, similarity(oc.at[win_oc[0], "Name_Key"], mn.at[win_mn[0], "Name_Key"]))
    return pairs

def run_matching(oc_frame, mn_frame, keys):
    oc_groups = dict(list(oc_frame.groupby(keys)))
    pairs = []
    for k, mn_g in mn_frame.groupby(keys):
        if k in oc_groups:
            pairs += match_seat(oc_groups[k], mn_g)
    mn_left = sorted(set(mn_frame.index) - {p[1] for p in pairs})
    return pairs, mn_left

In [7]:
keys = ["Year", "State", "Constituency_No"]
mn_general = myneta[~myneta["is_byelection"]].rename(columns={"year": "Year"})

g_pairs, g_mn_left = run_matching(results, mn_general, keys)
check = pd.DataFrame(g_pairs, columns=["oc", "mn", "rule", "score"])
print("OpenCity rows:", len(results), "| MyNeta rows:", len(mn_general), "| pairs:", len(g_pairs), "| MyNeta left:", len(g_mn_left))
print("pairs by rule:", check["rule"].value_counts().sort_index().to_dict())

# the winner flag comes from two different sources, so it is a fair test of the matching
agree = (results.loc[check["oc"], "Winner"].values == mn_general.loc[check["mn"], "is_winner"].values).mean()
print("winner flag agrees:", agree)

OpenCity rows: 10378 | MyNeta rows: 10228 | pairs: 9966 | MyNeta left: 262
pairs by rule: {1: 8963, 2: 151, 3: 82, 4: 517, 5: 195, 6: 49, 7: 9}
winner flag agrees: 1.0


In [8]:
mn_cols = {"candidate": "MN_Candidate", "party": "MN_Party", "education": "MN_Education", "age": "MN_Age",
           "criminal_cases": "MN_Criminal_Cases", "assets_rs": "Assets_Rs", "liabilities_rs": "Liabilities_Rs",
           "assets_status": "Assets_Status", "liabilities_status": "Liabilities_Status", "MN_Row": "MN_Row"}

def attach(oc_frame, mn_frame, pairs):
    link = pd.DataFrame(pairs, columns=["oc", "mn", "Match_Rule", "Match_Score"]).set_index("oc")
    mn_part = mn_frame.loc[link["mn"], list(mn_cols)].rename(columns=mn_cols).set_axis(link.index)
    out = oc_frame.join(link[["Match_Rule", "Match_Score"]]).join(mn_part)
    out["Assets_Nil"] = out["Assets_Status"].eq("nil").where(out["MN_Row"].notna())
    out["Has_MyNeta"] = out["MN_Row"].notna()
    out["Has_Assets"] = out["Assets_Rs"].notna()
    return out

general = attach(results, mn_general, g_pairs).rename(columns={"Year": "Election_Year"})
general["Term"] = general["Election_Year"]
general["Is_ByElection"] = False
general["OC_File"] = "opencity_results_clean"
print(general.groupby(["Election_Year", "State"])["Has_MyNeta"].mean().unstack().round(3))

State          Bihar  Maharashtra  Tamil Nadu  Uttar Pradesh
Election_Year                                               
2014           0.967        0.931       0.989          0.946
2019           0.952        0.912       0.955          0.939
2024           0.988        0.973       0.994          0.994


### By-elections

The OpenCity by-election file stops at 2021, so the 2022 (Azamgarh, Rampur, Mainpuri) and 2024 (Nanded) by-elections in MyNeta have nothing to match. NOTA is dropped, and position and margin are worked out again without it.

In [9]:
is_nota = oc_bye["Party"].eq("NOTA") | oc_bye["Candidate"].str.upper().isin(["NOTA", "NONE OF THE ABOVE"])
bye = oc_bye[oc_bye["State"].isin(STATES) & (oc_bye["Poll_No"] > 0) & ~is_nota].copy()
bye = add_name_keys(bye, "Candidate").rename(columns={"Year": "Bye_Year", "Valid_Votes": "Seat_Valid_Votes"})

# same seat names as the general table
seat_names = dict(zip(zip(seats["State"], seats["Constituency_No"]), seats["Constituency_Name"]))
bye["Constituency_Name"] = [seat_names[k] for k in zip(bye["State"], bye["Constituency_No"])]

seat_cols = ["State", "Constituency_No", "Bye_Year"]
bye["Position"] = bye.groupby(seat_cols)["Votes"].rank(method="min", ascending=False)
bye["Winner"] = bye["Position"] == 1
first = bye[bye["Position"] == 1].set_index(seat_cols)["Votes"]
second = bye[bye["Position"] == 2].set_index(seat_cols)["Votes"]
bye = bye.drop(columns="Margin").merge((first - second).rename("Margin"), left_on=seat_cols, right_index=True)
bye["Incumbent"] = bye["Incumbent"].map(lambda v: np.nan if pd.isna(v) else str(v) == "True")
assert bye.groupby(seat_cols)["Winner"].sum().eq(1).all()

mn_bye = myneta[myneta["is_byelection"]].copy()
mn_bye["Bye_Year"] = mn_bye["bye_election_date"].dt.year

b_pairs, b_mn_left = run_matching(bye, mn_bye, seat_cols)
oc_events = set(bye.groupby(seat_cols).groups)
print("by-election seats:", len(oc_events), "| rows:", len(bye), "| pairs:", len(b_pairs), "| MyNeta left:", len(b_mn_left))
print("only in MyNeta:", sorted(set(mn_bye.groupby(seat_cols).groups) - oc_events))

byelection = attach(bye, mn_bye, b_pairs)
byelection["Election_Year"] = byelection["Bye_Year"]
byelection["Term"] = np.where(byelection["Bye_Year"] <= 2018, 2014, 2019)
byelection["Is_ByElection"] = True
byelection["OC_File"] = "opencity_byelections_2014_onwards"
byelection["Party_Full"] = byelection["Party_Raw"]

by-election seats: 12 | rows: 134 | pairs: 118 | MyNeta left: 49
only in MyNeta: [('Maharashtra', 16, 2024), ('Uttar Pradesh', 7, 2022), ('Uttar Pradesh', 21, 2022), ('Uttar Pradesh', 69, 2022)]


### Combine

In [10]:
columns = ["OC_File", "OC_Row", "MN_Row", "State", "Constituency_No", "Constituency_Name", "Constituency_Type", "Term", "Election_Year",
           "Is_ByElection", "Bye_Year", "Person_ID", "Candidate", "Candidate_Key", "Namesake_In_Seat", "Sex", "Incumbent",
           "Party", "Party_Full", "Party_ID", "Party_Canonical", "Party_Type", "Party_Method", "Party_Group",
           "Position", "Winner", "Votes", "Vote_Share_Percentage", "Margin", "Seat_Valid_Votes",
           "MN_Candidate", "MN_Party", "MN_Education", "MN_Age", "MN_Criminal_Cases", "Assets_Rs", "Liabilities_Rs",
           "Assets_Status", "Liabilities_Status", "Assets_Nil", "Has_MyNeta", "Has_Assets", "Match_Rule", "Match_Score"]

INT_COLS = ["OC_Row", "MN_Row", "Constituency_No", "Term", "Election_Year", "Bye_Year", "Person_ID", "Party_ID", "Position",
            "Votes", "Margin", "Seat_Valid_Votes", "MN_Age", "MN_Criminal_Cases", "Match_Rule"]
FLOAT_COLS = ["Vote_Share_Percentage", "Assets_Rs", "Liabilities_Rs", "Match_Score"]

def set_types(frame):
    frame = frame.reindex(columns=columns)
    frame[INT_COLS] = frame[INT_COLS].apply(pd.to_numeric).round().astype("Int64")
    frame[FLOAT_COLS] = frame[FLOAT_COLS].astype("float64")
    for c in ["Namesake_In_Seat", "Incumbent", "Assets_Nil"]:
        frame[c] = frame[c].astype("boolean")
    return frame

general_out = set_types(general)
bye_out = set_types(byelection)
all_out = pd.concat([general_out, bye_out], ignore_index=True)

unmatched = pd.concat([
    mn_general.loc[g_mn_left].rename(columns={"Year": "year"}).assign(Reason="no partner in seat"),
    mn_bye.loc[b_mn_left].assign(Reason=["no partner in seat" if k in oc_events else "by-election not in OpenCity"
                                         for k in zip(mn_bye.loc[b_mn_left, "State"], mn_bye.loc[b_mn_left, "Constituency_No"], mn_bye.loc[b_mn_left, "Bye_Year"])]),
])[["MN_Row", "year", "State", "const", "Constituency_No", "candidate", "party", "Party_Canonical",
    "is_winner", "is_byelection", "bye_election_date", "assets_rs", "Reason"]]

print("general:", general_out.shape, "| by-election:", bye_out.shape, "| unmatched MyNeta:", len(unmatched))
print(unmatched["Reason"].value_counts().to_dict())

general: (10378, 44) | by-election: (134, 44) | unmatched MyNeta: 311
{'no partner in seat': 268, 'by-election not in OpenCity': 43}


### Checks

In [11]:
# no rows lost, no MyNeta row used twice
assert len(general_out) == len(results)
matched = all_out["MN_Row"].dropna()
assert matched.is_unique and len(matched) + len(unmatched) == len(myneta)

# one winner per seat and election
assert all_out.groupby(["Is_ByElection", "Election_Year", "State", "Constituency_No"])["Winner"].sum().eq(1).all()

# both sources agree on who won
m = all_out[all_out["Has_MyNeta"]]
assert (m["Winner"].values == myneta.loc[m["MN_Row"].astype(int), "is_winner"].values).all()

# MyNeta columns are empty only when there is no MyNeta partner
assert (all_out["Assets_Rs"].isna() == ~all_out["Has_MyNeta"]).all()

gw = general_out[general_out["Winner"]]
print("rows:", len(general_out), "general,", len(bye_out), "by-election | MyNeta matched", len(matched), "+ unmatched", len(unmatched), "=", len(myneta))
print("winners with MyNeta data:", int(gw["Has_MyNeta"].sum()), "of", len(gw))
print(gw.loc[~gw["Has_MyNeta"], ["Election_Year", "Constituency_Name", "Candidate", "Party"]].to_string(index=False))

rows: 10378 general, 134 by-election | MyNeta matched 10084 + unmatched 311 = 10395
winners with MyNeta data: 618 of 621
 Election_Year Constituency_Name                    Candidate Party
          2014 MUMBAI NORTH WEST GAJANAN CHANDRAKANT KIRTIKAR   SHS
          2014       FARRUKHABAD                MUKESH RAJPUT   BJP
          2019          AKBARPUR         DEVENDRA SINGH BHOLE   BJP


The three winners without MyNeta data are seats where the MyNeta page has no winner marked, so there is nothing to match them to.

In [12]:
general_out.to_csv(clean_dir / f"merged_general_{COLLECTION_DATE}.csv", index=False)
bye_out.to_csv(clean_dir / f"merged_byelection_{COLLECTION_DATE}.csv", index=False)
all_out.to_csv(clean_dir / f"merged_all_{COLLECTION_DATE}.csv", index=False)
unmatched.to_csv(clean_dir / f"merged_unmatched_myneta_{COLLECTION_DATE}.csv", index=False)
party_map.to_csv(clean_dir / f"party_map_{COLLECTION_DATE}.csv", index=False)

# read back to make sure nothing changed on the way
back = pd.read_csv(clean_dir / f"merged_all_{COLLECTION_DATE}.csv")
assert back.shape == all_out.shape
assert np.allclose(back["Assets_Rs"].fillna(-1), all_out["Assets_Rs"].fillna(-1))

In [13]:
# median assets (crore), winners vs the rest
view = general_out[general_out["Has_Assets"]].assign(Assets_Crore=lambda d: d["Assets_Rs"] / 1e7)
view.groupby(["Election_Year", "Winner"])["Assets_Crore"].agg(["size", "median"]).round(2).unstack()

size       median      
Winner        False True   False True 
Election_Year                         
2014           3271   205   0.21  3.39
2019           2912   206   0.23  6.58
2024           3165   207   0.27  9.37